# **Pathly**
## **Test Plan**
**Team Members**: Sammi Chong, Ellie Lu, Naseeha Masub, Saba Sohail @ Stony Brook University

---
### **Overview**
When agents code, they write tests alongside it. However, these tests do not simulate real edge cases and error-prone scenarios, contributing to the illusion of correctness. As such, manual testing is needed to ensure proper functionality. In this document, we discuss how and what we tested, what we found, and what is still untested.

---
### **Testing Approach**

| Approach | What it is | When we use it |
|---|---|---|
| **Automated testing** | Jest and React Native Testing Library for the app, and the Node test runner for the backend. Most of these tests are written by AI, so they supplement our other testing instead of replacing it. | Every pull request and every push to `main`, through CI |
| **Manual web testing** | We run the core user flows in a desktop browser, either on the hosted site or the local dev server. | Each new feature |
| **Manual phone testing** | We open the web app in Safari on a real iPhone. Desktop browsers don't show iOS touch or layout behavior, so this catches problems nothing else does. | Layout, scrolling, and touch changes |
| **Native testing** | We run the app on a phone through Expo Go or a development build. Expo Go can't show everything: on iOS it has no Google Maps, and it can't complete Google sign-in. | Checking that the native app runs |
| **Browser engine testing** | We run a web build in WebKit (the engine behind Safari) at several phone sizes to measure layouts. It can't reproduce real iOS scrolling or the real keyboard, so it doesn't replace testing on an iPhone. | Layout bugs that need before and after measurements |
| **Referencing** | We compare our data and features with Transit, Google Maps, and Citymapper. | Checking accuracy and expected behavior |

---
### **Automated Test Results**

| Suite | Location | Runner | Files | Tests | Result |
|---|---|---|---|---|---|
| Mobile | `mobile/__tests__/` | Jest + React Native Testing Library | 45 | 423 | All pass |
| Backend | `firebase/functions/test/` | Node test runner | 23 | 189 | All pass |
| **Total** | | | **68** | **612** | **All pass** |

Counts are from `main` at commit `505e88c`. CI runs lint, both test suites, the type check, and the build. To run everything locally from the repository root:

```sh
npm run lint
npm test
npm run typecheck
```

Backend tests replay recorded agency feeds from `firebase/functions/test/fixtures`, so delays, alerts, and missing trips behave the same on every run.

---
### **Manual Test Cases**

**Status key**
- **Pass**: checked by hand and worked as expected.
- **Automated**: covered by the automated suites, with no separate manual result recorded.
- **Partial**: works in the conditions we tried, with a limitation noted.
- **Not tested**: still needs to be run.

#### **Map and Location**

| ID | Test | Expected result | Status |
|---|---|---|---|
| MAP-1 | Pan past NYC and Long Island | The map returns to the service area | Pass |
| MAP-2 | Compare the map with Google Maps | Map data and locations match | Pass |
| MAP-3 | Open a route and view its path | Route geometry follows the real line | Pass |
| MAP-4 | Pan the home map, then let it rest | Nearby searches around the purple dot at the map center | Automated |
| MAP-5 | Press the location button after panning | The map returns to the rider and Nearby follows | Automated |
| MAP-6 | Deny location permission | The app falls back to Stony Brook and shows a notice | Automated |
| MAP-7 | Wait after the location notice appears | "Location unavailable" disappears after 1 second | Automated |
| MAP-8 | Rotate or tilt the map, then press the compass | The map turns back to north-up and flat | Automated |
| MAP-9 | Drag the home map in Expo Go on an iPhone | The map moves freely on Apple Maps | Not tested (see Known Issues) |

#### **Transit Data and Search**

| ID | Test | Expected result | Status |
|---|---|---|---|
| DATA-1 | Compare live and scheduled times with Transit and Google Maps | Times match the reference apps | Pass |
| DATA-2 | Compare vehicle locations with reference apps | Vehicles appear where expected | Pass |
| DATA-3 | Search for a line, stop, or address | Relevant results appear, with NYC and Long Island first | Pass |
| DATA-4 | View a route when live data is unavailable | The app shows an unavailable state rather than wrong data | Pass |
| DATA-5 | Open a slow-loading screen | A loading state appears until data arrives | Pass |
| DATA-6 | Plan a trip with no available transit | The app says no trips were found and offers a new time | Automated |
| DATA-7 | Read trip fares across agencies | Fares match each agency's rules | Automated |

#### **Trips and Navigation**

| ID | Test | Expected result | Status |
|---|---|---|---|
| NAV-1 | Plan a trip and open its details | Legs, transfers, and times are shown correctly | Pass |
| NAV-2 | Start a trip and follow it | Guidance moves through each step of the trip | Pass |
| NAV-3 | Restart the app during a trip | The trip can be resumed | Automated |
| NAV-4 | Finish a trip | The trip is added to Recents | Automated |

#### **Accounts and Saved Data**

| ID | Test | Expected result | Status |
|---|---|---|---|
| ACC-1 | Sign up and log in with email and password | The account opens the app | Pass |
| ACC-2 | Sign in with Google on the web | The Firebase popup signs the rider in | Pass |
| ACC-3 | Sign in with Google in a development build | The rider returns to the app signed in | Not tested |
| ACC-4 | Save a place, stop, trip, or route | It appears under Favorites and stays there | Pass |
| ACC-5 | Save a place and view the home map | A pin appears at the place | Pass |
| ACC-6 | Try to read another user's data | The request is denied | Automated |

#### **iOS Safari Layout and Scrolling**

| ID | Test | Expected result | Status |
|---|---|---|---|
| WEB-1 | Swipe on a fixed part of a screen | The screen itself doesn't move | Pass |
| WEB-2 | Scroll the Nearby list on Home | The list scrolls, starting on a card or between cards | Pass |
| WEB-3 | Scroll a route, fare, or navigation page | The sheet scrolls over the map | Pass |
| WEB-4 | Scroll the route results list | The list stays visible, with no white block in front of it | Pass |
| WEB-5 | Swipe future departures sideways | The row scrolls smoothly, like a normal horizontal scroll | Pass |
| WEB-6 | Scroll Settings | The list scrolls normally | Pass |
| WEB-7 | Check the top and bottom of the screen | No black borders appear around the app | Pass |
| WEB-8 | Open the keyboard on a screen with a text field | Containers stay in place after the keyboard closes | Partial: map height held in a simulated keyboard test in WebKit |
| WEB-9 | Focus a small text field | Safari doesn't zoom in and stay zoomed | Not tested |
| WEB-10 | View Home at 320, 390, 412, and 430 px wide | Nothing runs off the screen and the location notice isn't clipped | Partial: measured in WebKit for Home only |
| WEB-11 | Rotate the phone | The layout adjusts and stays usable | Not tested |
| WEB-12 | Repeat WEB-1 to WEB-7 in Chrome on Android | Same results as iOS Safari | Not tested |

---
### **Bugs Found Through Manual Testing**

The automated tests passed while every bug below was present, because they don't run on a real device or in Safari. This is why we don't rely on them alone.

| Bug | Cause | Fix |
|---|---|---|
| Whole screen scrolled in iOS Safari | Safari ignores `overflow: hidden` on the page | Custom `public/index.html` that only allows swipes on scrollable areas, text fields, and maps |
| Containers moved when the keyboard opened | Screens sized themselves from a height that shrinks with the keyboard, while the page didn't | Shared `useViewportSize` hook, based on the app's root element |
| Home, route, and fare pages wouldn't scroll on iOS | Exact cause in Safari not confirmed; these lists sit over a draggable map and ignore touches themselves | `useTouchScroll` hook follows the finger on iOS browsers |
| Departures row lagged | Touch swipes were scrolled a second time by code meant for mouse drags | Skip that code on touch screens |
| White block covered route results | A header bar sized from the trip card sat in front of the list | Turned the header off on that page |
| Location notice stayed forever | It was shown for as long as location was unavailable | Hide it after 1 second |
| Location notice clipped on small phones | It ran under the map buttons | Added side padding so it wraps |
| Blank map in Expo Go on iOS | Expo Go on iOS has no Google Maps SDK | Use Apple Maps in that one case |
| Web console error on route results | Stray spaces between two elements on one line | Moved the element onto its own line |

---
### **Known Issues and Limitations**

- **Home map dragging in Expo Go on iOS:** Apple Maps can fight the rider's drag because the map keeps resetting its own position. The fix hasn't been merged. Other platforms aren't affected, but MAP-9 stays open until it is.
- **Google sign-in on phones:** the button only appears when the platform's client ID is set, and Expo Go can't complete the sign-in. It needs a development build.
- **Map styling in Expo Go on iOS:** Apple Maps ignores our custom map style, so the styled map can only be checked on Android, the web, or a development build.
- **Finger scrolling on iOS:** the glide after a swipe comes from our own code, so it can feel slightly different from native scrolling.

---
### **Next Steps**

1. Run every **Not tested** case, starting with Android Chrome (WEB-12) and rotation (WEB-11).
2. Build an iOS development build to test Google sign-in and Google Maps on a real phone.
3. Merge the home map fix for Expo Go on iOS and rerun MAP-9.
4. Repeat the iOS Safari cases on the deployed site at https://pathly.web.app.
5. Add the bugs above to the automated suites where a test can reproduce them.